# Calculate country level population data on 0.25 resolution

In [1]:
import os
import json
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
SSP_scenario = "SSP2"

POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / SSP_scenario)
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")

In [3]:
# Use country mask to allign population data
mask_file = "country_id_mask_0.25.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
id_map = xr.open_dataset(mask_path)["country_id"]

# Load population data at 0.25 resolution
pop_file = f"{SSP_scenario.lower()}_era5_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)

# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

In [4]:
# Country IDs present in the map (drop NaN = no country)
ids = np.array(sorted(country_to_id.values()))

# Sum per country
country_sum = pop.groupby(id_map.rename("country")).sum()   # dims: (country,)
country_sum = country_sum.reindex(country=ids, fill_value=0)

In [6]:
# Save country level population
description = ("0.25deg country level population sum for years 2000-2100 "
               "- scripts by A.F. Wells (2026)")

country_sum.attrs["description"] = description
country_sum.attrs["SSP_scenario"] = SSP_scenario

out_file = f"{SSP_scenario.lower()}_era5_country_level_2000-2100.nc"
out_path = os.path.join(POP_DIR, out_file)
print(f"Saving to {out_path}")
country_sum.to_netcdf(out_path)

print("All processing complete.")

Saving to /glade/work/awells/workflow/SSP_pop/SSP2/ssp2_era5_country_level_2000-2100.nc
All processing complete.
